# NSOS — Sondagem da ATENÇÃO (recall): destravar o híbrido?

Investigação focada (branch `nsos-gpu-phases12`), motivada pelo genbench v5:
na sonda de **recall associativo** o ranking foi **mamba > híbrido > atenção**,
com a atenção travada perto do baseline (n_kv=8: attn **0.19** vs mamba **0.82**).

**Auditoria linha-a-linha do caminho VIVO no T4** (`batched_gqa_causal_attention_kernel`
+ `rope_apply`) confirmou: a matemática da atenção está **correta** (gradcheck +
paridade GPU passam) — a fraqueza de recall é **arquitetural/otimização, não bug**.
Suspeitos ranqueados (com evidência de código):
1. **GQA `n_kv=2`** — o genbench força; o default de produção é 4. 4 query heads
   colapsam em 2 espaços de K/V → menos capacidade de casamento-por-conteúdo.
2. **RoPE × `head_dim=32`** (128/4) — só ~metade dos 16 pares de frequência gira
   devagar o bastante p/ casamento invariante-a-posição. Copy (posicional)
   funciona, recall (conteúdo) não. Alavanca: `head_dim=64` via `n_heads=2`.
3. **LR / subajuste** — a loss da atenção cai monotônica mas não converge em 10k.

**Este notebook isola 1-3 com um sweep SÓ-CONFIG (sem mudar código):**
`GQA {n_kv 2 vs 4}` × `head_dim {32 vs 64}` × `LR {2e-3, 5e-3}`, em AR recall
(treina n_kv=8; avalia n_kv=8 e o mais difícil n_kv=16), com Mamba como **teto de
referência** e um veredito **pré-registrado e calibrado honestamente**.

⚠️ **Calibração honesta:** um Transformer que FUNCIONA resolve AR n_kv=8 ~**1.0** —
esse é o teto real, NÃO o nosso Mamba (0.82). O objetivo é ver quão perto a atenção
chega com os fixes de config, e se isso levanta o híbrido em direção ao Mamba.

Requer que o genbench v5 já tenha sido rodado ao menos uma vez (compartilha o
cache do build `.so`). Runtime → GPU (T4).

## 1 — GPU + (opcional) Drive p/ cache do build

In [ ]:
import os, subprocess
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/nsos_genbench'); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
except Exception as e:
    print('[drive] indisponivel (ok):', e); DRIVE_ROOT = Path('/content/nsos_genbench'); DRIVE_ROOT.mkdir(exist_ok=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=False)


## 2 — Clone da branch + prova de versão

In [ ]:
import os, subprocess
from pathlib import Path
REPO_URL = 'https://github.com/vitorGgC569/reimagined.git'
REPO_ROOT = Path('/content/reimagined'); BRANCH = 'nsos-gpu-phases12'
TOKEN = None
try:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'GH_TOKEN'):
        try: TOKEN = userdata.get(key)
        except Exception: TOKEN = None
        if TOKEN: break
except Exception: pass
url = REPO_URL.replace('https://', f'https://oauth2:{TOKEN}@') if TOKEN else REPO_URL
if REPO_ROOT.exists():
    subprocess.run(['git','-C',str(REPO_ROOT),'remote','set-url','origin',url], check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'fetch','--depth','1','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'reset','--hard',f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,url,str(REPO_ROOT)], check=True)
subprocess.run(['git','-C',str(REPO_ROOT),'remote','set-url','origin',REPO_URL], check=False)
os.environ['REPO_ROOT'] = str(REPO_ROOT)
sha = subprocess.run(['git','-C',str(REPO_ROOT),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip()
mamba = (REPO_ROOT/'OXN/nsos/src/mamba2.cpp').read_text('utf-8')
bitlin_h = (REPO_ROOT/'OXN/nsos/include/bitlinear.h').read_text('utf-8')
assert 'proper_selective_ssm' in mamba and 'forward_proper_step' in mamba, 'fonte stale (pre-v3)'
assert 'mamba_a_eff' in mamba, 'fonte stale: falta N1 (A reparametrizado em log-domain)'
assert 'qat_gpu_active_' in bitlin_h, 'fonte stale: falta K3 (QAT real em GPU no BitLinear)'
print(f'[ver] HEAD={sha} | Mamba proper + passo on-device + A em log-domain (N1) + GPU QAT (K3): OK')


## 3 — Build `nsos_ext` + gate de gradcheck (CUDA T4 sm_75)

In [ ]:
import os, sys, subprocess, shutil, glob
from pathlib import Path
if 'nsos_ext' in sys.modules:
    raise RuntimeError('nsos_ext ja carregado — Runtime > Restart.')
PYTAG = f'cp{sys.version_info.major}{sys.version_info.minor}'
sha = subprocess.run(['git','-C',str(REPO_ROOT),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip()
BUILD = str(REPO_ROOT / 'OXN/nsos/build-genbench')
EXT_DIR = Path('/content/nsos_ext_genbench'); EXT_DIR.mkdir(exist_ok=True)
CACHE_SO = DRIVE_ROOT / '_bootstrap' / f'nsos_ext_genbench.{PYTAG}.{sha}.so'
subprocess.run([sys.executable,'-m','pip','install','-q','pybind11','numpy'], check=True)
if CACHE_SO.exists():
    shutil.copy2(CACHE_SO, EXT_DIR/'nsos_ext.so'); os.environ['NSOS_BUILD_GENBENCH']=BUILD
    print('[build] .so do cache:', CACHE_SO)
else:
    r = subprocess.run(['cmake','-S',str(REPO_ROOT/'OXN/nsos'),'-B',BUILD,
                        '-DNSOS_ENABLE_CUDA=ON','-DCMAKE_CUDA_ARCHITECTURES=75',
                        '-DNSOS_BUILD_PYTHON=ON','-DNSOS_BUILD_TESTS=ON',
                        '-DNSOS_BUILD_CLI=OFF','-DNSOS_BUILD_API=OFF','-DNSOS_BUILD_OXTAMEM=OFF',
                        '-DCMAKE_BUILD_TYPE=Release',f'-DPython3_EXECUTABLE={sys.executable}'],
                       capture_output=True, text=True)
    print(r.stdout[-1000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-3000:]); raise RuntimeError('cmake configure falhou')
    GATE_TESTS = ['test_gradcheck', 'test_gpu_parity_mamba_proper',
                  'test_gpu_parity_mamba_proper_stream', 'test_gpu_parity_mamba_nstate']
    r = subprocess.run(['cmake','--build',BUILD,'-j','2','--target','nsos_ext',*GATE_TESTS],
                       capture_output=True, text=True)
    print(r.stdout[-1000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-3000:]); raise RuntimeError('build falhou')
    so = sorted(glob.glob(f'{BUILD}/**/nsos_ext*.so', recursive=True)); assert so
    shutil.copy2(so[0], EXT_DIR/'nsos_ext.so'); CACHE_SO.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(so[0], CACHE_SO); os.environ['NSOS_BUILD_GENBENCH']=BUILD
    print('[build] ok')
B = os.environ['NSOS_BUILD_GENBENCH']
# gradcheck (matematica do backward, FD vs hand-derived) + paridade GPU dos kernels
# tocados hoje (N1: A reparametrizado em log-domain; streaming batched do proper-SSM)
# -- corretude verificada SEM gastar tempo de treino, antes de qualquer comparacao
# de mixer abaixo.
GATE_FAIL = False
for name in ['test_gradcheck', 'test_gpu_parity_mamba_proper',
             'test_gpu_parity_mamba_proper_stream', 'test_gpu_parity_mamba_nstate']:
    found = False
    for cand in (f'{B}/{name}', f'{B}/Release/{name}', f'{B}/{name}.exe'):
        if os.path.exists(cand):
            found = True
            rr = subprocess.run([cand], capture_output=True, text=True)
            ok = rr.returncode == 0
            GATE_FAIL = GATE_FAIL or not ok
            print(f'[{name}]', 'PASS' if ok else f'FAIL (rc={rr.returncode})\n{rr.stdout[-1500:]}\n{rr.stderr[-1500:]}')
            break
    if not found:
        print(f'[{name}] binario nao encontrado (build pulou o alvo?)')
if GATE_FAIL:
    raise RuntimeError('gate de corretude falhou -- NAO prossiga para o treino sem investigar')


## 4 — Tarefas + avaliação

**AR:** prompt `[BOS k1 v1 ... kN vN SEP q]`, resposta `[valor de q]` (loss = só o
valor → 100% sinal de recall). `n_kv` = capacidade. **Selective copy:** campo com K
dados entre BLANK + MARK → resposta = os K dados; treina curto, testa longo.
`eval_*` sempre roda com `set_training_mode(False)` (caminho de inferência do
modelo — ternário 1.58-bit quando os pesos já passaram pela fase quantizada; float
se QAT está OFF nesta run, ver célula 5).

In [ ]:
import random, numpy as np

def make_ar(rng, n_kv, n_key, n_val):
    # vocabularios DISJUNTOS: chaves em [0,n_key), valores em [n_key,n_key+n_val).
    # (id ja diz se e chave ou valor -> circuito de recall limpo; era a causa do AR
    # nao treinar na v3, onde chave/valor compartilhavam o vocab.)
    BOS, SEP = n_key + n_val, n_key + n_val + 1
    keys = rng.sample(range(n_key), n_kv)
    vals = [n_key + rng.randrange(n_val) for _ in range(n_kv)]
    kv = dict(zip(keys, vals)); q = rng.choice(keys)
    prompt = [BOS]
    for k, v in zip(keys, vals): prompt += [k, v]
    prompt += [SEP, q]
    return prompt, [kv[q]]

def make_selcopy(rng, field_len, n_data, n_sym):
    BLANK, MARK, BOS = n_sym, n_sym + 1, n_sym + 2
    pos = sorted(rng.sample(range(field_len), n_data)); data = [rng.randrange(n_sym) for _ in range(n_data)]
    field = [BLANK] * field_len
    for p, d in zip(pos, data): field[p] = d
    return [BOS] + field + [MARK], data

def eval_ar(model, data, V):
    model.set_training_mode(False)
    cor = tot = 0
    for prompt, ans in data:
        lg = np.asarray(model.forward_ids(prompt).cpu().numpy()).reshape(-1, V)
        cor += int(int(np.argmax(lg[-1])) == ans[0]); tot += 1
    return cor / max(tot, 1)

def eval_copy(model, data, V):
    model.set_training_mode(False)
    cor = tot = 0
    for prompt, ans in data:
        seq = prompt + ans
        lg = np.asarray(model.forward_ids(seq).cpu().numpy()).reshape(-1, V)
        ok = all(int(np.argmax(lg[len(prompt) + j - 1])) == ans[j] for j in range(len(ans)))
        cor += int(ok); tot += 1
    return cor / max(tot, 1)

# Nota (v5): isto NAO e mais verdade incondicionalmente -- desde a auditoria de
# hoje, BitLinear tem um caminho real de QAT em GPU (fake-quant STE ternario+int8),
# entao quando NSOS_GEN_QAT=1 o forward de treino E o de eval passam pelos MESMOS
# pesos quantizados (sem mismatch). Com NSOS_GEN_QAT=0 (default aqui) o treino fica
# em float o tempo todo, igual as runs v1-v4 -- ver a celula 5.
print('[tasks] AR (vocab disjunto) + selective-copy definidos')


## 5 — Harness da sondagem (build por-condição + treino AR)

`build_attn_variant(variant, V, n_heads, n_kv)` — mesmo tamanho (d=128, 4
camadas), MoE/KAN/TTT OFF. QAT OFF (isola a arquitetura, float o tempo todo,
igual à comparação de mixer do genbench). `attn`: todas as camadas atenção;
`mamba`: todas SSM; `hybrid`: alternado.

In [ ]:
import os, sys, random, time
import numpy as np
os.environ['NSOS_MAMBA_PROPER_SSM']  = '1'
os.environ['NSOS_MAMBA_CONV_K']      = '3'
os.environ['NSOS_MAMBA_A_LOGSPACED'] = '1'
os.environ['NSOS_GPU_POOL']          = '1'
sys.path.insert(0, str(__import__('pathlib').Path('/content/nsos_ext_genbench')))
import nsos_ext as nsos
dev = nsos.Device.GPU

def build_attn_variant(variant, V, n_heads, n_kv, rope_theta=10000.0):
    c = nsos.ModelConfig()
    c.num_layers = 4; c.d_model = 128; c.vocab_size = V
    c.n_heads = n_heads; c.n_kv_heads = n_kv; c.max_context_tokens = 1024
    c.rope_theta = float(rope_theta)   # config path (portavel; env e instavel no Windows)
    c.use_exact_attention_training = True; c.use_flash_attn = False
    c.use_moe = False; c.use_kan = False; c.use_ttt = False; c.dropout = 0.0
    if variant == 'attn':
        c.attention_period = 1; c.attention_slot = 0      # 100% atencao
    elif variant == 'mamba':
        c.attention_period = 4; c.attention_slot = 4      # 0% atencao (so SSM)
    else:
        c.attention_period = 2; c.attention_slot = 1      # alternado
    m = nsos.JambaModel(c, dev); m.to(dev)
    return m

def train_ar(model, lr, steps, batch, n_kv_train, n_key, n_val, seed, label):
    tr = nsos.Trainer(model, lr); tr.warmup_steps = max(50, steps // 10)
    tr.total_training_steps = steps
    tr.phase_scheduler.progressive_qat_enabled = False   # isola: float o tempo todo
    tr.first_token_loss_scale = 1.0; tr.eos_loss_scale = 1.0
    rng = random.Random(seed * 991 + 7)
    model.set_training_mode(True)
    win = max(1, steps // 5); run = 0.0
    for s in range(steps):
        ps, ans = [], []
        for _ in range(batch):
            p, a = make_ar(rng, n_kv_train, n_key, n_val); ps.append(p); ans.append(a)
        run += tr.train_supervised_batch(ps, ans)
        if (s + 1) % win == 0:
            print(f'    [{label}] {s+1}/{steps} loss~{run/win:.3f}'); run = 0.0
    model.set_training_mode(False)

N_KEY = 16; N_VAL = 16; V = N_KEY + N_VAL + 2; base = 1.0 / N_VAL
KV_TEST = [8, 16]
STEPS = int(os.environ.get('NSOS_PROBE_STEPS', '6000'))
BATCH = int(os.environ.get('NSOS_PROBE_BATCH', '16'))
SEEDS = [int(x) for x in os.environ.get('NSOS_PROBE_SEEDS', '0,1').split(',')]
# (rotulo, variante, n_heads, n_kv, lr)
CONDS = [
    ('attn   hd32 GQA(nkv2) lr2e-3', 'attn',   4, 2, 2e-3),  # baseline (repro genbench)
    ('attn   hd32 MHA(nkv4) lr2e-3', 'attn',   4, 4, 2e-3),  # lever GQA
    ('attn   hd64 MHA(nh2)  lr2e-3', 'attn',   2, 2, 2e-3),  # lever head_dim (RoPE)
    ('attn   hd64 MHA(nh2)  lr5e-3', 'attn',   2, 2, 5e-3),  # head_dim + LR
    ('hybrid hd64 MHA(nh2)  lr2e-3', 'hybrid', 2, 2, 2e-3),  # payoff: levanta o hibrido?
    ('mamba  (teto ref)     lr2e-3', 'mamba',  4, 2, 2e-3),  # referencia
]
print(f'[cfg] d=128 L=4 | AR treina n_kv=8, avalia {KV_TEST} | steps={STEPS} batch={BATCH} seeds={SEEDS}')
print(f'[cfg] {len(CONDS)} condicoes x {len(SEEDS)} seeds = {len(CONDS)*len(SEEDS)} runs | QAT=OFF (float)')


## 6 — Sweep (AR recall) — treina n_kv=8, avalia n_kv=8 e 16

In [ ]:
res = {}; t0 = time.time()
for (label, variant, nh, nkv, lr) in CONDS:
    accs = {k: [] for k in KV_TEST}
    for seed in SEEDS:
        nsos.set_seed(seed)
        m = build_attn_variant(variant, V, nh, nkv)
        train_ar(m, lr, STEPS, BATCH, 8, N_KEY, N_VAL, seed, f'{label} s{seed}')
        ev = random.Random(seed * 13 + 5)
        for k in KV_TEST:
            d = [make_ar(ev, k, N_KEY, N_VAL) for _ in range(300)]
            accs[k].append(eval_ar(m, d, V))
        print(f'  [{label} s{seed}] ' + ' '.join(f'n_kv={k}:{accs[k][-1]:.2f}' for k in KV_TEST))
    res[label] = {k: (float(np.mean(accs[k])), float(np.std(accs[k]))) for k in KV_TEST}
print('\n' + '=' * 78)
print(f'ATTN SWEEP — AR recall (media+/-desvio, {len(SEEDS)} seeds; baseline={base:.3f})   ({time.time()-t0:.0f}s)')
print(f'{"condicao":<32}' + ''.join(f'{("n_kv="+str(k)):>18}' for k in KV_TEST))
for (label, *_ ) in CONDS:
    print(f'  {label:<30}' + ''.join(f'{res[label][k][0]:>11.3f}+/-{res[label][k][1]:.2f}' for k in KV_TEST))
print('=' * 78)


## 7 — VEREDITO (calibrado no teto real, não no Mamba)

In [ ]:
print('=' * 78)
attn_labels = [l for (l, v, *_ ) in CONDS if v == 'attn']
base_lbl = 'attn   hd32 GQA(nkv2) lr2e-3'
best_attn = max(attn_labels, key=lambda l: res[l][8][0])
mamba_lbl = 'mamba  (teto ref)     lr2e-3'
hyb_lbl   = 'hybrid hd64 MHA(nh2)  lr2e-3'
print('SWEEP DA ATENCAO — leitura:')
print(f'  baseline atencao (hd32 GQA): n_kv=8 = {res[base_lbl][8][0]:.3f}')
print(f'  MELHOR atencao ({best_attn.strip()}): n_kv=8 = {res[best_attn][8][0]:.3f}  (n_kv=16 = {res[best_attn][16][0]:.3f})')
print(f'  Mamba (teto interno):        n_kv=8 = {res[mamba_lbl][8][0]:.3f}  (n_kv=16 = {res[mamba_lbl][16][0]:.3f})')
print(f'  Hibrido c/ melhor-attn cfg:  n_kv=8 = {res[hyb_lbl][8][0]:.3f}  (n_kv=16 = {res[hyb_lbl][16][0]:.3f})')
gain = res[best_attn][8][0] - res[base_lbl][8][0]
print(f'\n  Ganho de config na atencao (melhor - baseline): +{gain:.3f}')
print('\nCALIBRACAO HONESTA: um Transformer que FUNCIONA resolve AR n_kv=8 ~1.0.')
print('  Esse e o teto REAL -- nao o nosso Mamba. Regua pre-registrada:')
if res[best_attn][8][0] >= 0.70:
    print(f'  -> DESTRAVOU por CONFIG: melhor atencao {res[best_attn][8][0]:.2f} >= 0.70.')
    print('     GQA/head_dim eram o gargalo. Proximo: usar essa cfg no hibrido/producao.')
elif res[best_attn][8][0] >= 0.45:
    print(f'  -> PARCIAL: {res[best_attn][8][0]:.2f} (subiu de {res[base_lbl][8][0]:.2f} mas < 0.70).')
    print('     Config ajuda mas nao basta -> alavancas de CODIGO: RoPE base theta maior')
    print('     e/ou QK-norm (aceleram induction heads). Implementar sob flag + gradcheck.')
else:
    print(f'  -> NAO destravou por config ({res[best_attn][8][0]:.2f}). O teto e mais fundo:')
    print('     provavelmente RoPE-vs-recall estrutural -> testar NoPE-em-alguns-heads / QK-norm')
    print('     (mudancas de codigo), ou aceitar schedule Mamba-pesado no hibrido.')
print('\nOBS: mesmo destravada, a atencao pode ficar < Mamba nesta escala minuscula')
print('(o estado SSD e memoria associativa direta). "Superar tudo" precisa de escala real,')
print('nao de sonda sintetica -- isto aqui decide "vale escalar", nao "e SOTA".')
print('=' * 78)


## 8 — Sweep de RoPE `theta` (a atenção melhora com mais dims lentos?)

Teste da hipótese RoPE: `theta` maior → mais pares de frequência giram devagar →
mais dimensões por-head **invariantes a posição** para casamento por conteúdo
(recall). **Ressalva honesta:** o par de maior frequência (i=0) gira ~1 rad/token
independente de `theta` — então `theta` grande **não** vira NoPE, só **adiciona
dims lentos**. Como dobrar `head_dim` (que também adiciona dims lentos) rendeu só
+0.03, a expectativa aqui é BAIXA — mas o teste é barato e fecha a questão. Roda
na melhor arquitetura de atenção do sweep anterior (hd64 MHA) + no híbrido.

In [ ]:
THETAS = [float(x) for x in os.environ.get('NSOS_PROBE_THETAS', '10000,1e6,1e8').split(',')]
theta_res = {}; t0 = time.time()
# (rotulo, variante, n_heads, n_kv, lr) -- fixa a melhor arq de atencao (hd64 MHA)
THETA_CONDS = [
    ('attn   hd64 MHA(nh2)', 'attn',   2, 2, 2e-3),
    ('hybrid hd64 MHA(nh2)', 'hybrid', 2, 2, 2e-3),
]
for (label, variant, nh, nkv, lr) in THETA_CONDS:
    for th in THETAS:
        accs = {k: [] for k in KV_TEST}
        for seed in SEEDS:
            nsos.set_seed(seed)
            m = build_attn_variant(variant, V, nh, nkv, rope_theta=th)
            train_ar(m, lr, STEPS, BATCH, 8, N_KEY, N_VAL, seed, f'{label} theta={th:.0e} s{seed}')
            ev = random.Random(seed * 13 + 5)
            for k in KV_TEST:
                d = [make_ar(ev, k, N_KEY, N_VAL) for _ in range(300)]
                accs[k].append(eval_ar(m, d, V))
            print(f'  [{label} theta={th:.0e} s{seed}] ' + ' '.join(f'n_kv={k}:{accs[k][-1]:.2f}' for k in KV_TEST))
        theta_res[(label, th)] = {k: (float(np.mean(accs[k])), float(np.std(accs[k]))) for k in KV_TEST}
print('\n' + '=' * 78)
print(f'ROPE THETA SWEEP — AR recall (media+/-desvio, {len(SEEDS)} seeds; baseline={base:.3f})   ({time.time()-t0:.0f}s)')
print(f'{"cond / theta":<30}' + ''.join(f'{("n_kv="+str(k)):>18}' for k in KV_TEST))
for (label, variant, nh, nkv, lr) in THETA_CONDS:
    for th in THETAS:
        r = theta_res[(label, th)]
        print(f'  {label+" th="+format(th,".0e"):<28}' + ''.join(f'{r[k][0]:>11.3f}+/-{r[k][1]:.2f}' for k in KV_TEST))
print('=' * 78)


## 9 — VEREDITO theta

In [ ]:
print('=' * 78)
attn_th = {th: theta_res[('attn   hd64 MHA(nh2)', th)][8][0] for th in THETAS}
best_th = max(THETAS, key=lambda th: attn_th[th])
base_th = attn_th.get(10000.0, attn_th[min(THETAS)])
print('ROPE THETA (atencao hd64 MHA), recall n_kv=8:')
for th in THETAS:
    print(f'  theta={th:.0e}: {attn_th[th]:.3f}')
print(f'\n  melhor theta = {best_th:.0e} -> {attn_th[best_th]:.3f}  (baseline theta=1e4 -> {base_th:.3f})')
delta = attn_th[best_th] - base_th
print(f'  ganho de theta: +{delta:.3f}')
print('\nLEITURA:')
if attn_th[best_th] >= 0.70:
    print('  -> RoPE ERA o gargalo: theta grande destravou. Fix principal = rope_theta maior')
    print('     (e/ou NoPE-em-alguns-heads). Adotar no config da atencao.')
elif delta >= 0.10:
    print('  -> RoPE contribui (ganho real mas nao resolve). Combinar theta + QK-norm (codigo).')
else:
    print('  -> RoPE NAO e o gargalo principal (ganho <0.10, como o head_dim ja sugeria).')
    print('     A causa e TRAINABILITY do induction head -> proxima alavanca = QK-norm')
    print('     (normalizar q,k por head antes do dot; acelera formacao de induction heads),')
    print('     ou aceitar que o Mamba e o mixer de recall e usar schedule Mamba-pesado.')
print('=' * 78)
